## 1. Load & Inspect Data

In [ ]:
import pandas as pd

df = pd.read_csv('cleaned_canada.csv')
df.head()
df.info()  # check dtypes and non-null counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44896 entries, 0 to 44895
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   City            44896 non-null  object 
 1   Province        44896 non-null  object 
 2   Latitude        44896 non-null  float64
 3   Longitude       44896 non-null  float64
 4   Price           44896 non-null  float64
 5   Bedrooms        44896 non-null  float64
 6   Bathrooms       44896 non-null  float64
 ...  (23 columns total)

## 2. Missing Values

In [ ]:
df.isnull().sum()  # check for empty cells

City                  0
Province              0
Latitude              0
Longitude             0
Price                 0
Bedrooms              0
Bathrooms             0
Acreage               0
Property Type         0
Square Footage        0
Garage                0
Parking               0
Basement          29934
Exterior          27457
Fireplace             0
Heating            6654
Flooring          29680
Roof              34844
Waterfront            0
Sewer                 0
Pool               0

In [ ]:
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_pct.sort_values(ascending=False)
# Roof, Basement, Flooring, Exterior are more than half empty -> better to drop the columns
# Heating is only ~15% missing -> worth keeping / handling separately

Roof              77.610478
Basement          66.674091
Flooring          66.108339
Exterior          61.156896
Heating           14.820919
Province           0.000000
City               0.000000
... (remaining columns 0%)

## 3. Target Variable: Price Distribution & Outliers

In [ ]:
df['Price'].describe()  # check price distribution

count    4.489600e+04
mean     1.070457e+06
std      1.442961e+06
min      5.000000e+04
25%      3.999900e+05
50%      6.880000e+05
75%      1.200000e+06
max      5.880000e+07
Name: Price, dtype: float64

# std (1.44M) > mean (1.07M) -> unusual signal, values are spread very wide
# max is ~85x the median -> a small number of extremely expensive listings are skewing things

In [ ]:
import matplotlib.pyplot as plt

df['Price'].hist(bins=50)
plt.xlabel('Price')
plt.ylabel('House count')
plt.title('House price distribution')
plt.show()
# hard to read -- the long tail of expensive listings compresses everything else

<Figure size 640x480 with 1 Axes>

In [ ]:
# Zoom in below $3M to see the bulk of the distribution
df[df['Price'] < 3_000_000]['Price'].hist(bins=50)
plt.xlabel('Price')
plt.ylabel('House Count')
plt.title('House price distribution (under $3M)')
plt.show()

<Figure size 640x480 with 1 Axes>

In [ ]:
# Log transform as an alternative way to view the skewed distribution
import numpy as np

np.log(df['Price']).hist(bins=50)
plt.xlabel('log(Price)')
plt.ylabel('House Count')
plt.title('House price distribution (log scale)')
plt.show()

<Figure size 640x480 with 1 Axes>

In [ ]:
# 99th percentile check -> use as an outlier cutoff
df['Price'].quantile(0.99)

np.float64(6500000.0)

In [ ]:
# Remove the top 1% most expensive listings as outliers
df = df[df['Price'] <= 6_500_000]
print(df.shape)

(44450, 23)

## 4. Cleaning

Dropped `Roof`, `Basement`, `Flooring`, `Exterior`. 
`Heating` was also dropped here, favoring simplicity over squeezing out a partially-populated feature.

In [ ]:
df = df.drop(columns=['Roof', 'Basement', 'Flooring', 'Exterior', 'Heating'])
print(df.isnull().sum())

City              0
Province          0
Latitude          0
Longitude         0
Price             0
Bedrooms          0
Bathrooms         0
Acreage           0
Property Type     0
Square Footage    0
Garage            0
Parking           0
Fireplace         0
Waterfront        0
Sewer             0
Pool              0
Garden            0
Balcony           0
dtype: int64

In [ ]:
# Check category cardinality before deciding how to encode each column
print(df['City'].nunique())
print(df['Province'].nunique())
print(df['Property Type'].nunique())

3149
11
9

`City` has 3,149 unique values — too many for one-hot encoding. Dropped in favor of `Latitude`/`Longitude` (already numeric) and `Province` (11 categories, manageable).

In [ ]:
df = df.drop(columns=['City'])
print(df['Waterfront'].unique())

['No' 'Yes']

## 5. Feature Engineering

In [ ]:
binary_cols = ['Garage', 'Parking', 'Fireplace', 'Sewer', 'Pool', 'Garden', 'Balcony']

for col in binary_cols:
    print(col, df[col].unique())
# Sewer turns out to have 5 categories, not Yes/No -> needs one hot encoding instead

Garage ['Yes' 'No']
Parking ['Yes' 'No']
Fireplace ['No' 'Yes']
Sewer ['municipal' 'septic' 'none' 'alternative' 'private']
Pool ['No' 'Yes']
Garden ['No' 'Yes']
Balcony ['No' 'Yes']

In [ ]:
binary_cols = ['Garage', 'Parking', 'Fireplace', 'Pool', 'Garden', 'Balcony', 'Waterfront']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df[binary_cols].head()

   Garage  Parking  Fireplace  Pool  Garden  Balcony  Waterfront
0       1        1          0     0       0        0           0
1       1        1          0     0       0        0           0
2       0        0          0     0       0        0           0
3       1        0          1     0       0        0           0
4       1        1          0     0       0        0           0

In [ ]:
df = pd.get_dummies(df, columns=['Sewer'], prefix='Sewer')
df.head()

In [ ]:
df = pd.get_dummies(df, columns=['Property Type', 'Province'], prefix=['PropType', 'Province'])
print(df.shape)

(44450, 39)

In [ ]:
df.info()
print(df.dtypes.value_counts())

<class 'pandas.core.frame.DataFrame'>
Index: 44450 entries, 0 to 44895
Data columns (total 39 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Latitude                    44450 non-null  float64
 1   Longitude                   44450 non-null  float64
 2   Price                       44450 non-null  float64
 ...  (39 columns total)

In [ ]:
X = df.drop(columns=['Price'])
y = df['Price']

# A small number of rows had NaN left in Garden/Balcony after mapping -> filled with 0
X['Garden'] = X['Garden'].fillna(0)
X['Balcony'] = X['Balcony'].fillna(0)

## 6. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)

(35560, 38)
(8890, 38)

## 7. Baseline Model: Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({'actual price': y_test, 'predicted price': y_pred})
comparison.head(10)

       actual price  predicted price
27490      319900.0     9.659965e+04
37180      399900.0     8.640491e+05
41420      349900.0     3.288763e+05
32335      629900.0     9.610058e+05
30540      198000.0     6.144982e+03
13762      749000.0     7.861897e+05
887       2338000.0     1.620221e+06
5352       839900.0     5.126235e+05
26598      487900.0     4.409385e+05
5682       359900.0     1.564051e+05

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: ${mae:,.0f}")
print(f"R²: {r2:.3f}")

MAE: $365,295
R²: 0.214

In [ ]:
# Which features got the largest weights?
coefficients = pd.DataFrame({
    'col': X_train.columns,
    'coef': model.coef_
})
coefficients = coefficients.sort_values('coef', ascending=False)
print(coefficients.head(10))

                       col          coef
31             Province_NL  1.147258e+06
35             Province_PE  8.083341e+05
30             Province_NB  7.278719e+05
32             Province_NS  6.706701e+05
24  PropType_Single Family  3.104255e+05
34             Province_ON  2.847927e+05
20         PropType_Duplex  2.478476e+05
9               Waterfront  2.087352e+05
23    PropType_MultiFamily  1.544643e+05
13       Sewer_alternative  1.489927e+05

`Province_NL` shows a coefficient of $1.1M, which isn't plausible — Newfoundland & Labrador is not one of Canada's most expensive markets. 
Because every one-hot encoded category was kept (instead of dropping one as a reference), the columns are perfectly collinear (they always sum to 1), which destabilizes the linear model's coefficients. A fix for next time: re-encode with `drop_first=True` on `Province`, `Property Type`, and `Sewer`.

In [ ]:
residuals = y_test - y_pred

plt.scatter(y_test, residuals, alpha=0.3)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Actual Price')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Prediction Error by Price Range')
plt.show()
# Errors widen for higher-priced homes -- a sign the linear model can't capture non-linear pricing patterns

<Figure size 640x480 with 1 Axes>

## 8. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest MAE: ${mae_rf:,.0f}")
print(f"Random Forest R²: {r2_rf:.3f}")

Random Forest MAE: $166,646
Random Forest R²: 0.860

In [ ]:
importances = pd.DataFrame({
    'col': X_train.columns,
    'importance': rf_model.feature_importances_
})
importances = importances.sort_values('importance', ascending=False)
print(importances.head(10))

                       col  importance
5           Square Footage    0.438263
1                Longitude    0.334528
0                 Latitude    0.093908
4                  Acreage    0.067080
3                Bathrooms    0.018301
2                 Bedrooms    0.012632
24  PropType_Single Family    0.005838
8                Fireplace    0.003975
6                   Garage    0.003808
7                  Parking    0.003464

## 9. Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf_model, X, y, cv=5, scoring='r2')
print("5-fold R² scores:", scores)
print("Mean R²:", scores.mean())
print("Std:", scores.std())

5-fold R² scores: [0.77631341 0.76890507 0.81769921 0.42813735 0.64941366]
Mean R²: 0.6880937404823134
Std: 0.14155156909767094

In [ ]:
# One fold scored far lower than the rest -- check whether the data is sorted
print(df.index[:10])
print(df.index[-10:])

Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype='int64')
Index([44886, 44887, 44888, 44889, 44890, 44891, 44892, 44893, 44894, 44895], dtype='int64')

In [ ]:
from sklearn.model_selection import KFold

# Shuffle before splitting into folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_shuffled = cross_val_score(rf_model, X, y, cv=kf, scoring='r2')
print("5-fold R² scores:", scores_shuffled)
print("Mean R²:", scores_shuffled.mean())
print("Std:", scores_shuffled.std())

5-fold R² scores: [0.85899754 0.85256426 0.87144124 0.86413896 0.86642371]
Mean R²: 0.8627131446561205
Std: 0.0064619880532999375

Always check whether a dataset is sorted before cross-validating. `cross_val_score`'s default `cv=5` does not shuffle, so a dataset sorted by region (as this one likely was when scraped) can concentrate certain patterns into a single fold and produce a misleadingly unstable score. Enabling `shuffle=True` in `KFold` tightened the score spread from a std of 0.142 to 0.006.

## 10. Summary

Linear Regression (baseline)
- MAE : $365,295
- R2 : 0.214

Random Forest
- MAE : **$166,646**
- R2 : **0.860** (single split) / **0.863** (5-fold CV)

The final Random Forest model explains about 86% of the variance in Canadian house prices in this dataset, with an average error of roughly $167K. The consistency across 5-fold cross-validation (std = 0.006) confirms this performance is stable rather than a lucky split.
